# 📊 60秒ニュース速報 / Flash News — チャンネル分析 + Claude対話

このノートブックは:
1. **YouTube Data API v3** で自分のチャンネルの動画統計を取得
2. **Claude Opus 4.7** にデータと質問を投げて分析・改善案をもらう

## 初回セットアップ
**Colab左サイドバー → 🔑「シークレット」** で以下を登録:
- `ANTHROPIC_API_KEY` （`.env` から）
- `YT_API_KEY`（Google Cloud Console → YouTube Data API v3 → 認証情報 → APIキー）
  - ※自分のチャンネルの **公開データ** だけならAPIキーで十分
  - 非公開データやアップロードしたい場合は OAuth（client_secret + token）必要

## 使い方
順にセル実行 → 一番下の `chat("質問")` で対話

In [ ]:
!pip install -q anthropic google-api-python-client pandas matplotlib seaborn

In [ ]:
# === API キー読み込み（Colab Secrets 推奨） ===
import os
try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    YT_API_KEY = userdata.get('YT_API_KEY')
except Exception:
    # ローカル/別環境用フォールバック
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY') or input('ANTHROPIC_API_KEY: ')
    YT_API_KEY = os.environ.get('YT_API_KEY') or input('YT_API_KEY: ')

# === チャンネル ID（自分のチャンネル）===
# YouTube Studio → 設定 → チャンネル → 詳細設定 → チャンネルID で取得
CHANNEL_ID = 'YOUR_CHANNEL_ID_HERE'  # ← ここを書き換える
print(f'Anthropic key: {ANTHROPIC_API_KEY[:15]}...')
print(f'YouTube key  : {YT_API_KEY[:15]}...')
print(f'Channel      : {CHANNEL_ID}')

In [ ]:
# === YouTube Data API ヘルパー ===
from googleapiclient.discovery import build
import pandas as pd
from datetime import datetime, timezone

yt = build('youtube', 'v3', developerKey=YT_API_KEY)

def fetch_channel_videos(channel_id: str, max_results: int = 50) -> pd.DataFrame:
    '''チャンネルの最新動画と統計を取得'''
    # チャンネルのアップロードプレイリストID取得
    ch = yt.channels().list(part='contentDetails,snippet,statistics', id=channel_id).execute()
    if not ch.get('items'):
        raise ValueError(f'チャンネルが見つかりません: {channel_id}')
    uploads = ch['items'][0]['contentDetails']['relatedPlaylists']['uploads']
    ch_title = ch['items'][0]['snippet']['title']
    ch_subs = ch['items'][0]['statistics'].get('subscriberCount', '?')
    ch_views = ch['items'][0]['statistics'].get('viewCount', '?')
    print(f'📺 {ch_title}: {ch_subs}人登録 / 総再生 {ch_views} 回')

    # アップロードプレイリストから動画ID取得
    video_ids = []
    page_token = None
    while len(video_ids) < max_results:
        resp = yt.playlistItems().list(
            playlistId=uploads, part='contentDetails', maxResults=50, pageToken=page_token
        ).execute()
        video_ids.extend(it['contentDetails']['videoId'] for it in resp['items'])
        page_token = resp.get('nextPageToken')
        if not page_token:
            break
    video_ids = video_ids[:max_results]

    # 動画詳細取得（50ずつバッチ）
    rows = []
    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i+50]
        resp = yt.videos().list(
            id=','.join(batch),
            part='snippet,statistics,contentDetails'
        ).execute()
        for v in resp['items']:
            sn, st = v['snippet'], v['statistics']
            published = datetime.fromisoformat(sn['publishedAt'].replace('Z', '+00:00'))
            age_h = (datetime.now(timezone.utc) - published).total_seconds() / 3600
            views = int(st.get('viewCount', 0))
            rows.append({
                'video_id': v['id'],
                'title': sn['title'],
                'published': published,
                'age_hours': round(age_h, 1),
                'views': views,
                'likes': int(st.get('likeCount', 0)),
                'comments': int(st.get('commentCount', 0)),
                'duration': v['contentDetails']['duration'],
                'views_per_hour': round(views / max(age_h, 0.5), 2),
                'url': f"https://youtu.be/{v['id']}",
            })
    df = pd.DataFrame(rows).sort_values('published', ascending=False).reset_index(drop=True)
    return df

df = fetch_channel_videos(CHANNEL_ID, max_results=50)
print(f'📊 {len(df)} 本の動画データを取得')
df.head(10)

In [ ]:
# === 概要統計 + ビジュアル ===
import matplotlib.pyplot as plt
import seaborn as sns

print('=== 直近10本のサマリー ===')
print(f"平均再生数 : {df['views'].mean():.0f}")
print(f"中央値再生 : {df['views'].median():.0f}")
print(f"最高再生   : {df['views'].max():.0f} ← {df.loc[df['views'].idxmax(), 'title']}")
print(f"最低再生   : {df['views'].min():.0f}")
print(f"平均CTR代理(views/h): {df['views_per_hour'].mean():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df.head(20).iloc[::-1].plot(x='title', y='views', kind='barh', ax=axes[0], legend=False, color='steelblue')
axes[0].set_title('最新20本の再生数'); axes[0].set_xlabel('views')
axes[1].scatter(df['age_hours'], df['views'], alpha=0.6)
axes[1].set_xlabel('経過時間 (h)'); axes[1].set_ylabel('views'); axes[1].set_title('時間 vs 再生数')
plt.tight_layout(); plt.show()

In [ ]:
# === Claude チャット関数（履歴付き） ===
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# 会話履歴（チャンネル分析の文脈を保持）
_history = []

def chat(question: str, include_data: bool = True) -> str:
    '''Claudeに質問。include_data=True なら現在の df をコンテキストに自動注入。'''
    global _history

    # 初回のみシステムにチャンネルデータを入れる
    system_parts = [
        'あなたは YouTube Shorts ニュース系チャンネルのプロデューサーです。',
        '視聴回数・タイトル傾向・公開時刻などのデータからバズ要因を抽出し、',
        '改善提案は具体的（タイトル例・サムネ案・投稿時刻など）に行う。',
        'ユーザーは日本語で質問するので日本語で答える。',
    ]
    if include_data and len(_history) == 0:
        # 初回だけデータを丸ごと入れる
        data_csv = df.head(50).to_csv(index=False)
        system_parts.append(f'\n=== チャンネルの直近50本データ ===\n{data_csv}')
    system = '\n'.join(system_parts)

    _history.append({'role': 'user', 'content': question})
    msg = client.messages.create(
        model='claude-opus-4-7',
        max_tokens=2000,
        system=system,
        messages=_history,
    )
    answer = msg.content[0].text
    _history.append({'role': 'assistant', 'content': answer})
    print(answer)
    return answer

def reset_chat():
    global _history
    _history = []
    print('履歴をクリアしました')

## 💬 ここから対話 - 質問例

In [ ]:
chat('直近50本のデータを見て、再生数が伸びたTOP3とその共通点を教えて。タイトルパターン・公開時刻・話題ジャンルの観点で。')

In [ ]:
chat('逆に再生数が伸びなかったボトム3の原因を分析して、次回避けるべきパターンを3つ提案して。')

In [ ]:
chat('現状のタイトル形式を見て、もっとバズりそうな新しいタイトル定型を5つ提案して。具体例も付けて。')

In [ ]:
# 自由質問はここで
chat('')